In [3]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve
from pykrx import stock
import time
import os
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 설정 및 계정과목 매핑 (Contains 로직 적용)
# ==============================================================================

DATA_FOLDER = "./" 

# [규칙 정의] (표준계정명, 검색키워드, 재무제표구분)
# 재무제표구분(sj_div): BS(재무상태표), IS(손익계산서), CF(현금흐름표/자본변동표 포함)
# 키워드: 해당 단어가 '포함'되면 매핑 (단, 우선순위 유의: 위에서부터 매칭되면 덮어씌워질 수 있음 -> 구체적인 것 먼저?)
# 여기서는 하나씩 순회하며 매핑하되, 중복 시 마지막 매핑이 적용됨. 
# 하지만 contains 로직이므로, '매출채권'을 검색하면 '장기매출채권'도 걸림.
ACCOUNT_RULES = [
    # --- 재무상태표 (BS) ---
    ('Total_Assets', '자산총계', 'BS'),
    ('Total_Assets', '총자산', 'BS'),
    ('Total_Liabilities', '부채총계', 'BS'),
    ('Total_Liabilities', '총부채', 'BS'),
    ('Total_Equity', '자본총계', 'BS'),
    ('Total_Equity', '총자본', 'BS'),
    ('Current_Assets', '유동자산', 'BS'),
    ('Current_Liabilities', '유동부채', 'BS'),
    ('Non_Current_Liabilities', '비유동부채', 'BS'),
    ('Non_Current_Liabilities', '고정부채', 'BS'),
    
    # 매출채권: '매출채권' 단어가 들어간 BS 항목
    ('Receivables', '매출채권', 'BS'), 
    
    ('Inventory', '재고자산', 'BS'),
    ('Inventory', '재고', 'BS'), # '재고'가 들어간 자산
    ('PPE', '유형자산', 'BS'),
    ('Accumulated_Depreciation', '감가상각누계액', 'BS'),
    ('Accumulated_Depreciation', '감가상각누계', 'BS'),

    # --- 손익계산서 (IS) ---
    ('Revenue', '매출액', 'IS'), # 수익(매출액) 등 포함
    ('Revenue', '영업수익', 'IS'),
    ('COGS', '매출원가', 'IS'),
    ('COGS', '영업비용', 'IS'), # 매출원가가 없고 영업비용만 있는 경우 대비
    ('Gross_Profit', '매출총이익', 'IS'),
    ('Operating_Income', '영업이익', 'IS'),
    ('Net_Income', '당기순이익', 'IS'),
    ('SG&A', '판매비와관리비', 'IS'),
    ('SG&A', '판관비', 'IS'),
    ('Interest_Expense', '이자비용', 'IS'), # '이자비용' 단어 포함
    ('Interest_Expense', '금융비용', 'IS'),

    # --- 현금흐름표 (CF) ---
    ('CFO', '영업활동', 'CF'), # 영업활동현금흐름, 영업활동으로인한현금흐름
    ('Depreciation_CF', '감가상각비', 'CF'), # 현금흐름표 상 상각비

    # --- 이익잉여금 (BS/IS/SCE 혼용 가능, 주로 BS나 자본변동표) ---
    # 이익잉여금은 재무상태표(BS)나 자본변동표(SCE)에 등장
    ('Retained_Earnings', '이익잉여금', 'BS'),
    ('Retained_Earnings', '결손금', 'BS'),
    ('Retained_Earnings', '미처분이익잉여금', 'BS'),
    ('Retained_Earnings', '미처리결손금', 'BS')
]

# ==============================================================================
# 2. 데이터 로드 및 전처리 (헤더 자동 탐색 및 컬럼 표준화 기능 추가)
# ==============================================================================

def normalize_columns(df):
    """
    다양한 칼럼명(한글/영어 혼용)을 분석에 필요한 표준 칼럼명으로 변경
    """
    # 1. 모든 칼럼명 소문자 변환 및 공백 제거
    df.columns = [str(c).strip().lower() for c in df.columns]
    
    # 2. 칼럼명 매핑
    rename_map = {
        # 계정명 관련
        '계정명': 'account_nm',
        '항목명': 'account_nm',
        'account_name': 'account_nm',
        'account': 'account_nm',
        
        # 금액 관련
        '당기': 'thstrm_amount',
        'thstrm_amount': 'thstrm_amount',
        'amount': 'thstrm_amount',
        '금액': 'thstrm_amount',
        
        # 연도 관련
        '사업연도': 'bsns_year',
        'year': 'bsns_year',
        
        # 구분 관련
        '구분': 'sj_div',
        'sj_div': 'sj_div'
    }
    
    df.rename(columns=rename_map, inplace=True)
    return df

def find_header_and_load(df_raw):
    """
    데이터프레임에서 '계정명'이나 'account'가 있는 행을 찾아 헤더로 재설정
    """
    # 1. 현재 헤더가 이미 올바른지 확인 (정상 케이스)
    sample_cols = [str(c).lower() for c in df_raw.columns]
    if any(x in str(sample_cols) for x in ['계정명', 'account', '항목명', 'account_nm']):
        return df_raw

    # 2. 헤더가 아닐 경우, 상위 10개 행을 탐색하여 진짜 헤더 찾기
    for i in range(min(10, len(df_raw))):
        row_values = df_raw.iloc[i].astype(str).tolist()
        # 해당 행에 '계정명' 등의 키워드가 있는지 검사
        if any(('계정명' in val or 'account' in val.lower()) for val in row_values):
            # i번째 행을 헤더로 설정
            new_header = df_raw.iloc[i]
            df_new = df_raw[i+1:].copy()
            df_new.columns = new_header
            return df_new
            
    # 못 찾으면 원본 반환 (어차피 이후 로직에서 걸러짐)
    return df_raw

def pivot_data(df):
    if df.empty: return pd.DataFrame()
    
    # [NEW 1] 헤더 위치 자동 보정
    df = find_header_and_load(df)
    
    # [NEW 2] 컬럼명 표준화 적용
    df = normalize_columns(df)
    
    # 필수 칼럼('account_nm')이 없으면 처리 불가 -> 빈 DF 반환
    if 'account_nm' not in df.columns:
        return pd.DataFrame()
    
    # 1. 계정과목 매핑 ('Contains' 로직)
    df['Std_Account'] = np.nan
    
    # sj_div(재무제표 구분)가 없는 경우 처리
    if 'sj_div' not in df.columns:
        df['sj_div'] = 'ALL'
    
    # 매핑 루프
    for std_acc, keyword, div_cond in ACCOUNT_RULES:
        # 조건: (계정명에 keyword 포함) AND (sj_div가 해당 구분)
        cond_keyword = df['account_nm'].astype(str).str.contains(keyword, na=False)
        
        if div_cond == 'CF':
            cond_div = df['sj_div'].astype(str).str.contains('CF', case=False) | \
                       df['sj_div'].astype(str).str.contains('현금', na=False)
        elif div_cond == 'BS':
            cond_div = df['sj_div'].isin(['BS', '재무상태표'])
        elif div_cond == 'IS':
            cond_div = df['sj_div'].isin(['IS', '손익계산서', '포괄손익계산서', 'CIS'])
        else:
            cond_div = True 

        mask = cond_keyword & cond_div
        df.loc[mask, 'Std_Account'] = std_acc
    
    # 매핑된 행만 추출
    df_clean = df.dropna(subset=['Std_Account'])
    
    # 2. 금액 컬럼 숫자 변환
    if 'thstrm_amount' in df_clean.columns:
        if df_clean['thstrm_amount'].dtype == object:
            df_clean['thstrm_amount'] = (df_clean['thstrm_amount'].astype(str)
                                         .str.replace(',', '')
                                         .str.replace('(', '-')
                                         .str.replace(')', '')
                                         .astype(float))
    else:
        # 금액 컬럼조차 없으면 빈 DF
        return pd.DataFrame()
    
    # 3. 피벗 (bsns_year 기준)
    if 'bsns_year' not in df_clean.columns:
        return pd.DataFrame()
        
    return df_clean.pivot_table(index='bsns_year', columns='Std_Account', values='thstrm_amount', aggfunc='first')

def load_financial_data(filepath):
    try:
        # read_excel 시 header=None 옵션을 쓰지 않고 일단 읽은 뒤 find_header_and_load에서 처리
        xls = pd.ExcelFile(filepath)
        df_fin = pd.DataFrame()
        
        # 연결 -> 별도 순 탐색
        target_sheets = ['CFS_Annual', 'OFS_Annual']
        
        for sheet in target_sheets:
            if sheet in xls.sheet_names:
                df_temp = pd.read_excel(xls, sheet_name=sheet)
                df_pivot = pivot_data(df_temp)
                
                if not df_pivot.empty:
                    df_fin = df_pivot
                    break
        
        if df_fin.empty:
            return pd.DataFrame()

        return df_fin.reset_index()

    except Exception as e:
        # zip file error 등 파일 손상 시
        return pd.DataFrame()

def load_local_stock_volatility(filepath):
    """
    로컬 엑셀 파일의 'Stock_Price' 시트를 읽어 연간 변동성 계산
    """
    try:
        if filepath.endswith('.xlsx'):
            df_price = pd.read_excel(filepath, sheet_name='Stock_Price')
        else:
            df_price = pd.read_csv(filepath)
            
        if df_price.empty:
            return pd.DataFrame()
            
        # 날짜/종가 컬럼 찾기 (대소문자 구분 없이)
        cols_lower = [c.lower() for c in df_price.columns]
        
        # Date 찾기
        date_candidates = [c for c in df_price.columns if 'date' in c.lower() or '일자' in c]
        if not date_candidates: return pd.DataFrame()
        date_col = date_candidates[0]
        
        # Close 찾기
        close_candidates = [c for c in df_price.columns if 'close' in c.lower() or '종가' in c]
        if not close_candidates: return pd.DataFrame()
        close_col = close_candidates[0]
        
        df_price[date_col] = pd.to_datetime(df_price[date_col])
        df_price['Year'] = df_price[date_col].dt.year
        
        # 수익률 및 변동성 계산
        df_price['Return'] = df_price.groupby('Year')[close_col].pct_change()
        volatility = df_price.groupby('Year')['Return'].std() * np.sqrt(252)
        
        return volatility.reset_index().rename(columns={'Year': 'bsns_year', 'Return': 'Volatility'})
        
    except Exception as e:
        # print(f"   !! 주가 데이터(로컬) 로드 실패: {e}")
        return pd.DataFrame()

def get_pykrx_market_cap(ticker, start_year, end_year):
    """
    Pykrx를 이용해 연말 시가총액만 가져옴
    """
    results = []
    ticker = str(ticker).zfill(6)
    
    print(f"   >> 시가총액 수집 중 ({start_year}~{end_year})...")
    
    for year in range(start_year, end_year + 1):
        try:
            ohlcv = stock.get_market_ohlcv(str(year)+"0101", str(year)+"1231", ticker)
            if ohlcv is None or ohlcv.empty:
                continue
            
            last_date = ohlcv.index[-1]
            last_date_str = last_date.strftime("%Y%m%d")
            
            cap_df = stock.get_market_cap(last_date_str, last_date_str, ticker)
            
            if not cap_df.empty:
                # 컬럼명 '시가총액' 확인
                mc = np.nan
                if '시가총액' in cap_df.columns:
                    mc = cap_df['시가총액'].iloc[0]
                results.append({'bsns_year': year, 'Market_Cap': mc})
                
            time.sleep(0.05)
        except:
            continue
            
    return pd.DataFrame(results)

def ensure_columns(df):
    # 피벗 후 없는 컬럼 NaN 생성
    required = set([rule[0] for rule in ACCOUNT_RULES])
    for col in required:
        if col not in df.columns:
            df[col] = np.nan
    return df

# ==============================================================================
# 3. Feature 계산 (수정된 로직 반영)
# ==============================================================================
def calculate_features(df):
    df = ensure_columns(df)
    df = df.copy()
    
    # 전기 데이터 생성
    cols = [c for c in df.columns if c not in ['bsns_year', 'Market_Cap', 'Volatility']]
    for c in cols:
        df[f'prev_{c}'] = df[c].shift(1)

    # 1. 감가상각비 대용치 (CF 감가상각비 -> BS 누계액 차분 -> 0)
    df['Dep_Proxy'] = df['Accumulated_Depreciation'] - df['prev_Accumulated_Depreciation']
    df['Dep_Proxy'] = df['Dep_Proxy'].apply(lambda x: x if (pd.notnull(x) and x > 0) else 0)
    
    df['Final_Dep'] = df['Depreciation_CF'].fillna(df['Dep_Proxy']).fillna(0)
    df['prev_Final_Dep'] = df['Final_Dep'].shift(1)

    # 2. 순자산 증가율 (부호 보정)
    df['F1_Equity_Growth'] = (df['Total_Equity'] - df['prev_Total_Equity']) / df['prev_Total_Equity'].abs()
    
    # 3. 기타 재무비율
    df['F1_Retained_Earnings_Ratio'] = df['Retained_Earnings'] / df['Total_Assets']
    df['F1_ROA'] = df['Net_Income'] / df['Total_Assets']
    df['F1_Debt_Ratio'] = df['Total_Liabilities'] / df['Total_Assets']
    df['F1_Current_Ratio'] = df['Current_Assets'] / df['Current_Liabilities']
    df['F1_ROE'] = df['Net_Income'] / df['Total_Equity']
    df['F1_Interest_Coverage'] = df['Operating_Income'] / df['Interest_Expense'].replace(0, np.nan)

    # 4. KMV 부도거리
    r, T = 0.035, 1
    # 변동성: 로컬 파일에서 구한 값 사용, 없으면 60% Penalty
    df['Vol_Adj'] = df['Volatility'].fillna(method='ffill').fillna(0.6)
    
    dd_list = []
    for idx, row in df.iterrows():
        try:
            E = row['Market_Cap']
            sigma_E = row['Vol_Adj']
            D = row['Total_Liabilities']
            
            if pd.isna(E) or pd.isna(D) or E <= 0 or D < 0:
                dd_list.append(np.nan)
                continue
            
            STD = row.get('Current_Liabilities', D*0.5)
            LTD = row.get('Non_Current_Liabilities', D*0.5)
            if pd.isna(STD): STD = D*0.5
            if pd.isna(LTD): LTD = D*0.5
            
            def equations(p):
                V, sigma_V = p
                d1 = (np.log(V/D) + (r + 0.5*sigma_V**2)*T) / (sigma_V*np.sqrt(T))
                d2 = d1 - sigma_V*np.sqrt(T)
                return (V*norm.cdf(d1) - D*np.exp(-r*T)*norm.cdf(d2) - E,
                        V*norm.cdf(d1)*sigma_V - E*sigma_E)
            
            V, sigma_V = fsolve(equations, (E+D, sigma_E*0.5))
            DP = STD + 0.5*LTD
            DD = (V - DP) / (V * sigma_V)
            dd_list.append(DD)
        except:
            dd_list.append(np.nan)
            
    df['F2_KMV_DD'] = dd_list

    # 5. Altman Z-Score
    X1 = (df['Current_Assets'] - df['Current_Liabilities']) / df['Total_Assets']
    X2 = df['Retained_Earnings'] / df['Total_Assets']
    X3 = df['Operating_Income'] / df['Total_Assets']
    X4 = df['Market_Cap'] / df['Total_Liabilities']
    X5 = df['Revenue'] / df['Total_Assets']
    df['F3_Z_Score'] = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5

    # 6. Beneish M-Score
    eps = 1e-6
    dsri = (df['Receivables']/(df['Revenue']+eps)) / (df['prev_Receivables']/(df['prev_Revenue']+eps) + eps)
    
    gp = df['Gross_Profit'].fillna(df['Revenue'] - df['COGS'])
    prev_gp = df['prev_Gross_Profit'].fillna(df['prev_Revenue'] - df['prev_COGS'])
    gmi = (prev_gp/(df['prev_Revenue']+eps)) / (gp/(df['Revenue']+eps) + eps)
    
    aq = 1 - (df['Current_Assets'] + df['PPE']) / (df['Total_Assets']+eps)
    prev_aq = 1 - (df['prev_Current_Assets'] + df['prev_PPE']) / (df['prev_Total_Assets']+eps)
    aqi = aq / (prev_aq + eps)
    
    sgi = df['Revenue'] / (df['prev_Revenue'] + eps)
    
    dep_rate = df['Final_Dep'] / (df['PPE'] + df['Final_Dep'] + eps)
    prev_dep_rate = df['prev_Final_Dep'] / (df['prev_PPE'] + df['prev_Final_Dep'] + eps)
    depi = prev_dep_rate / (dep_rate + eps)
    
    sgai = (df['SG&A']/(df['Revenue']+eps)) / (df['prev_SG&A']/(df['prev_Revenue']+eps) + eps)
    
    lvgi = (df['Total_Liabilities']/(df['Total_Assets']+eps)) / (df['prev_Total_Liabilities']/(df['prev_Total_Assets']+eps) + eps)
    
    tata = (df['Net_Income'] - df['CFO']) / (df['Total_Assets'] + eps)
    
    df['F4_M_Score'] = -4.84 + 0.920*dsri + 0.528*gmi + 0.404*aqi + 0.892*sgi + 0.115*depi - 0.172*sgai + 4.679*tata - 0.327*lvgi

    return df

# ==============================================================================
# 4. 메인 실행 함수
# ==============================================================================
def main():
    # 파일 로드 (파일명 확인)
    try:
        df_distressed_list = pd.read_csv('관리종목_상장폐지_업종포함.csv') 
        df_normal_list = pd.read_csv('master_table_cleaned(2).csv')
    except Exception as e:
        print(f"리스트 파일 로드 실패: {e}")
        return

    # 리스트 통합
    df_distressed_list['Target_Code'] = df_distressed_list['종목코드'].apply(lambda x: str(x).zfill(6))
    df_distressed_list['Company_Name'] = df_distressed_list['회사명']
    df_distressed_list['Status'] = 'Distressed'
    
    df_normal_list['Target_Code'] = df_normal_list['종목코드'].apply(lambda x: str(x).zfill(6))
    df_normal_list['Company_Name'] = df_normal_list['종목명']
    df_normal_list['Status'] = 'Normal'
    
    all_companies = pd.concat([
        df_distressed_list[['Target_Code', 'Company_Name', 'Status']], 
        df_normal_list[['Target_Code', 'Company_Name', 'Status']]
    ])
    
    print(f"총 {len(all_companies)}개 기업 분석 시작...\n")
    final_results = []
    
    for idx, row in all_companies.iterrows():
        code = row['Target_Code']
        name = row['Company_Name']
        status = row['Status']
        
        # 파일 경로 설정
        if status == 'Distressed':
            file_name = f"{code}_재무제표_1999_2025.xlsx"
        else:
            file_name = f"{code}_재무제표_2000_2025.xlsx"
            if not os.path.exists(os.path.join(DATA_FOLDER, file_name)):
                file_name = f"{code}_{name}.xlsx"
        
        full_path = os.path.join(DATA_FOLDER, file_name)
        
        if not os.path.exists(full_path):
            continue
            
        print(f"[{idx+1}/{len(all_companies)}] 처리 중: {name} ({code})")
        
        # 1. 재무 데이터 로드
        df_fin = load_financial_data(full_path)
        if df_fin.empty:
            continue
            
        # 2. 로컬 주가 변동성 로드 (파일 내 Stock_Price 시트 활용)
        df_vol = load_local_stock_volatility(full_path)
        if not df_vol.empty:
            df_fin = pd.merge(df_fin, df_vol, on='bsns_year', how='left')
        else:
            df_fin['Volatility'] = np.nan
            
        # 3. 시가총액 수집 (Pykrx)
        min_year = int(df_fin['bsns_year'].min())
        max_year = int(df_fin['bsns_year'].max())
        
        df_cap = get_pykrx_market_cap(code, min_year, max_year)
        
        if not df_cap.empty:
            df_fin = pd.merge(df_fin, df_cap, on='bsns_year', how='left')
        else:
            df_fin['Market_Cap'] = np.nan
            
        # 4. Feature 계산
        df_calculated = calculate_features(df_fin)
        
        # 메타데이터 추가
        df_calculated['Company_Code'] = code
        df_calculated['Company_Name'] = name
        df_calculated['Status'] = status
        
        final_results.append(df_calculated)
        
    if final_results:
        df_final = pd.concat(final_results, ignore_index=True)
        save_name = 'Bankruptcy_Prediction_Features_Final_V2.csv'
        df_final.to_csv(save_name, index=False, encoding='utf-8-sig')
        print(f"\n✅ 모든 작업 완료! '{save_name}' 파일이 저장되었습니다.")
    else:
        print("\n❌ 결과 데이터가 없습니다.")

if __name__ == "__main__":
    main()

총 3984개 기업 분석 시작...

[1/3984] 처리 중: 인피니트헬스케어 (071200)
   >> 시가총액 수집 중 (2015~2024)...
[2/3984] 처리 중: 신한글로벌액티브리츠 (481850)
   >> 시가총액 수집 중 (2024~2025)...
[3/3984] 처리 중: 코아스 (071950)
   >> 시가총액 수집 중 (2015~2024)...
[4/3984] 처리 중: 서희건설 (035890)
   >> 시가총액 수집 중 (2015~2024)...
[5/3984] 처리 중: 아이큐어 (175250)
   >> 시가총액 수집 중 (2018~2024)...
[6/3984] 처리 중: 하이드로리튬 (101670)
   >> 시가총액 수집 중 (2017~2022)...
[7/3984] 처리 중: 에스엘에스바이오 (246250)
   >> 시가총액 수집 중 (2019~2024)...
[8/3984] 처리 중: 코엔텍 (029960)
   >> 시가총액 수집 중 (2015~2018)...
[9/3984] 처리 중: 부산주공 (005030)
   >> 시가총액 수집 중 (2015~2024)...
[10/3984] 처리 중: 동성제약 (002210)
   >> 시가총액 수집 중 (2015~2024)...
[11/3984] 처리 중: 아이엠 (101390)
   >> 시가총액 수집 중 (2015~2024)...
[12/3984] 처리 중: 이엔플러스 (074610)
   >> 시가총액 수집 중 (2015~2024)...
[13/3984] 처리 중: 하이로닉 (149980)
   >> 시가총액 수집 중 (2015~2024)...
[14/3984] 처리 중: 스타에스엠리츠 (204210)
   >> 시가총액 수집 중 (2022~2024)...
[15/3984] 처리 중: 셀레스트라 (352770)
   >> 시가총액 수집 중 (2020~2024)...
[16/3984] 처리 중: 제일엠앤에스 (412540)
   >> 시가총액 수집 중 (2024~2

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve
from pykrx import stock
import time
import os
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 설정 및 계정과목 매핑
# ==============================================================================

DATA_FOLDER = "./" 
OUTPUT_FILE = 'Bankruptcy_Prediction_Features_Final_V2.csv' 

# 계정과목 매핑 규칙
ACCOUNT_RULES = [
    # --- 재무상태표 (BS) ---
    ('Total_Assets', '자산총계', 'BS'), ('Total_Assets', '총자산', 'BS'),
    ('Total_Liabilities', '부채총계', 'BS'), ('Total_Liabilities', '총부채', 'BS'),
    ('Total_Equity', '자본총계', 'BS'), ('Total_Equity', '총자본', 'BS'),
    ('Current_Assets', '유동자산', 'BS'), ('Current_Liabilities', '유동부채', 'BS'),
    ('Non_Current_Liabilities', '비유동부채', 'BS'), ('Non_Current_Liabilities', '고정부채', 'BS'),
    ('Receivables', '매출채권', 'BS'), 
    ('Inventory', '재고자산', 'BS'), ('Inventory', '재고', 'BS'), 
    ('PPE', '유형자산', 'BS'),
    ('Accumulated_Depreciation', '감가상각누계액', 'BS'), ('Accumulated_Depreciation', '감가상각누계', 'BS'),

    # --- 손익계산서 (IS) ---
    ('Revenue', '매출액', 'IS'), ('Revenue', '영업수익', 'IS'), ('Revenue', '수익(매출액)', 'IS'),
    ('COGS', '매출원가', 'IS'), ('COGS', '영업비용', 'IS'), 
    ('Gross_Profit', '매출총이익', 'IS'),
    ('Operating_Income', '영업이익', 'IS'), ('Operating_Income', '영업손실', 'IS'), # 영업손실도 영업이익 항목임
    ('Net_Income', '당기순이익', 'IS'), ('Net_Income', '당기순손실', 'IS'),
    ('SG&A', '판매비와관리비', 'IS'), ('SG&A', '판관비', 'IS'),
    ('Interest_Expense', '이자비용', 'IS'), ('Interest_Expense', '금융비용', 'IS'),

    # --- 현금흐름표 (CF) ---
    ('CFO', '영업활동', 'CF'), 
    ('Depreciation_CF', '감가상각비', 'CF'), 

    # --- 이익잉여금 ---
    ('Retained_Earnings', '이익잉여금', 'BS'), ('Retained_Earnings', '결손금', 'BS'),
    ('Retained_Earnings', '미처분이익잉여금', 'BS'), ('Retained_Earnings', '미처리결손금', 'BS')
]

# ==============================================================================
# 2. 데이터 로드 및 전처리 (개선됨)
# ==============================================================================

def normalize_columns(df):
    """ 컬럼명 표준화 """
    df.columns = [str(c).strip().lower() for c in df.columns]
    rename_map = {
        '계정명': 'account_nm', '항목명': 'account_nm', 'account_name': 'account_nm', 'account': 'account_nm', 'item_name': 'account_nm',
        '당기': 'thstrm_amount', 'thstrm_amount': 'thstrm_amount', 'amount': 'thstrm_amount', '금액': 'thstrm_amount',
        '사업연도': 'bsns_year', 'year': 'bsns_year',
        '구분': 'sj_div', 'sj_div': 'sj_div'
    }
    df.rename(columns=rename_map, inplace=True)
    return df

def find_header_and_load(df_raw):
    """ 
    헤더가 첫 행이 아닐 수 있으므로, '계정명' 같은 키워드가 있는 행을 찾아 헤더로 설정 
    """
    # 1. 이미 헤더가 정상인 경우
    sample_cols = [str(c).lower() for c in df_raw.columns]
    keywords = ['계정명', 'account', '항목명', 'account_nm', 'item']
    
    if any(k in str(sample_cols) for k in keywords):
        return df_raw

    # 2. 상위 15개 행 탐색
    for i in range(min(15, len(df_raw))):
        row_values = df_raw.iloc[i].astype(str).tolist()
        # 해당 행에 키워드가 포함되어 있는지 확인
        if any(k in str(row_values).lower() for k in keywords):
            new_header = df_raw.iloc[i]
            df_new = df_raw[i+1:].copy()
            df_new.columns = new_header
            return df_new
            
    return df_raw

def pivot_data(df):
    if df.empty: return pd.DataFrame()
    
    # 헤더 찾기 및 컬럼 표준화
    df = find_header_and_load(df)
    df = normalize_columns(df)
    
    # 필수 컬럼 확인
    if 'account_nm' not in df.columns: 
        return pd.DataFrame()
    
    # sj_div가 없으면 ALL로 통일
    if 'sj_div' not in df.columns: 
        df['sj_div'] = 'ALL'
    
    df['Std_Account'] = np.nan
    
    # 계정 매핑
    for std_acc, keyword, div_cond in ACCOUNT_RULES:
        cond_keyword = df['account_nm'].astype(str).str.contains(keyword, na=False)
        
        if div_cond == 'CF':
            cond_div = df['sj_div'].astype(str).str.contains('CF', case=False) | df['sj_div'].astype(str).str.contains('현금', na=False)
        elif div_cond == 'BS':
            cond_div = df['sj_div'].isin(['BS', '재무상태표'])
        elif div_cond == 'IS':
            cond_div = df['sj_div'].isin(['IS', '손익계산서', '포괄손익계산서', 'CIS'])
        else:
            cond_div = True 
            
        mask = cond_keyword & cond_div
        df.loc[mask, 'Std_Account'] = std_acc
    
    # 매핑된 데이터만 남김
    df_clean = df.dropna(subset=['Std_Account'])
    
    # 금액 컬럼 전처리
    if 'thstrm_amount' in df_clean.columns:
        if df_clean['thstrm_amount'].dtype == object:
            df_clean['thstrm_amount'] = (df_clean['thstrm_amount'].astype(str)
                                         .str.replace(',', '')
                                         .str.replace('(', '-')
                                         .str.replace(')', '')
                                         .apply(pd.to_numeric, errors='coerce'))
    else:
        return pd.DataFrame()
    
    # 사업연도 컬럼 확인
    if 'bsns_year' not in df_clean.columns: 
        return pd.DataFrame()
        
    # 피벗
    return df_clean.pivot_table(index='bsns_year', columns='Std_Account', values='thstrm_amount', aggfunc='first')

def load_financial_data(filepath):
    """
    [핵심 수정] 시트 이름을 지정하지 않고, 파일 내의 모든 시트를 순회하며 데이터를 찾습니다.
    """
    try:
        xls = pd.ExcelFile(filepath)
        df_fin = pd.DataFrame()
        
        # 엑셀 파일의 모든 시트를 순회
        for sheet in xls.sheet_names:
            try:
                df_temp = pd.read_excel(xls, sheet_name=sheet)
                df_pivot = pivot_data(df_temp)
                
                # 유효한 데이터(피벗 결과)가 나오면 병합하고 루프 중단 (또는 계속 병합할 수도 있음)
                # 여기서는 가장 데이터가 많은 시트를 쓰거나, 단순히 첫 번째 유효 시트를 씀
                if not df_pivot.empty:
                    # 필요한 주요 컬럼(자산, 매출 등)이 포함되어 있는지 확인하면 더 정확함
                    if 'Total_Assets' in df_pivot.columns or 'Revenue' in df_pivot.columns:
                        df_fin = df_pivot
                        # print(f"   -> 시트 발견: {sheet}") # 디버깅용
                        break
            except:
                continue
                
        if df_fin.empty: 
            return pd.DataFrame()
            
        return df_fin.reset_index()
        
    except Exception as e:
        return pd.DataFrame()

def load_local_stock_volatility(filepath):
    try:
        # 엑셀인 경우 Stock_Price 시트 우선, 없으면 첫번째 시트 시도
        if filepath.endswith('.xlsx'):
            try:
                df_price = pd.read_excel(filepath, sheet_name='Stock_Price')
            except:
                return pd.DataFrame() # Stock_Price 시트가 없으면 포기 (혹은 다른 로직 추가 가능)
        else:
            df_price = pd.read_csv(filepath)
            
        if df_price.empty: return pd.DataFrame()
        
        cols_lower = [str(c).lower() for c in df_price.columns]
        
        # Date 찾기
        date_col = next((c for c in df_price.columns if 'date' in str(c).lower() or '일자' in str(c)), None)
        # Close 찾기
        close_col = next((c for c in df_price.columns if 'close' in str(c).lower() or '종가' in str(c)), None)
        
        if not date_col or not close_col: return pd.DataFrame()
        
        df_price[date_col] = pd.to_datetime(df_price[date_col], errors='coerce')
        df_price.dropna(subset=[date_col], inplace=True)
        df_price['Year'] = df_price[date_col].dt.year
        
        # 수익률 및 변동성 계산
        df_price[close_col] = pd.to_numeric(df_price[close_col], errors='coerce')
        df_price['Return'] = df_price.groupby('Year')[close_col].pct_change()
        volatility = df_price.groupby('Year')['Return'].std() * np.sqrt(252)
        
        return volatility.reset_index().rename(columns={'Year': 'bsns_year', 'Return': 'Volatility'})
    except:
        return pd.DataFrame()

def get_pykrx_market_cap(ticker, start_year, end_year):
    results = []
    ticker = str(ticker).zfill(6)
    # print(f" ⏳ 시총 수집...", end="")
    
    # 너무 오래된 연도나 미래 연도는 제외
    if start_year < 2000: start_year = 2000
    if end_year > 2025: end_year = 2025
    
    for year in range(start_year, end_year + 1):
        try:
            # 연말 기준 (휴장일 고려하여 안전하게 12/20~12/31 범위의 마지막 거래일 찾기)
            # 단순히 1231로 조회하면 휴장일일 경우 데이터가 안올 수 있음 -> get_market_cap은 날짜 자동 보정 기능이 약함
            # get_market_ohlcv를 통해 연말 마지막 거래일을 확인
            ohlcv = stock.get_market_ohlcv(str(year)+"1201", str(year)+"1231", ticker)
            
            if ohlcv is None or ohlcv.empty:
                continue
                
            last_date = ohlcv.index[-1]
            last_date_str = last_date.strftime("%Y%m%d")
            
            cap_df = stock.get_market_cap(last_date_str, last_date_str, ticker)
            
            if not cap_df.empty:
                mc = np.nan
                if '시가총액' in cap_df.columns: 
                    mc = cap_df['시가총액'].iloc[0]
                results.append({'bsns_year': year, 'Market_Cap': mc})
            
            time.sleep(0.1) # 차단 방지
        except:
            continue
            
    return pd.DataFrame(results)

def ensure_columns(df):
    required = set([rule[0] for rule in ACCOUNT_RULES])
    for col in required:
        if col not in df.columns: df[col] = np.nan
    return df

# ==============================================================================
# 3. Feature 계산 함수
# ==============================================================================
def calculate_features(df):
    df = ensure_columns(df)
    df = df.copy()
    
    # 정렬 확인
    df = df.sort_values(by='bsns_year')
    
    cols = [c for c in df.columns if c not in ['bsns_year', 'Market_Cap', 'Volatility']]
    for c in cols:
        df[f'prev_{c}'] = df[c].shift(1)

    df['Dep_Proxy'] = df['Accumulated_Depreciation'] - df['prev_Accumulated_Depreciation']
    df['Dep_Proxy'] = df['Dep_Proxy'].apply(lambda x: x if (pd.notnull(x) and x > 0) else 0)
    df['Final_Dep'] = df['Depreciation_CF'].fillna(df['Dep_Proxy']).fillna(0)
    df['prev_Final_Dep'] = df['Final_Dep'].shift(1)

    # 0으로 나누기 방지
    def safe_div(a, b):
        return np.where(b == 0, np.nan, a / b)

    df['F1_Equity_Growth'] = safe_div((df['Total_Equity'] - df['prev_Total_Equity']), df['prev_Total_Equity'].abs())
    df['F1_Retained_Earnings_Ratio'] = safe_div(df['Retained_Earnings'], df['Total_Assets'])
    df['F1_ROA'] = safe_div(df['Net_Income'], df['Total_Assets'])
    df['F1_Debt_Ratio'] = safe_div(df['Total_Liabilities'], df['Total_Assets'])
    df['F1_Current_Ratio'] = safe_div(df['Current_Assets'], df['Current_Liabilities'])
    df['F1_ROE'] = safe_div(df['Net_Income'], df['Total_Equity'])
    df['F1_Interest_Coverage'] = safe_div(df['Operating_Income'], df['Interest_Expense'])

    # KMV
    r, T = 0.035, 1
    df['Vol_Adj'] = df['Volatility'].fillna(method='ffill').fillna(0.6)
    
    dd_list = []
    for idx, row in df.iterrows():
        try:
            E = row['Market_Cap']
            sigma_E = row['Vol_Adj']
            D = row['Total_Liabilities']
            
            if pd.isna(E) or pd.isna(D) or E <= 0 or D < 0:
                dd_list.append(np.nan)
                continue
            
            STD = row.get('Current_Liabilities', D*0.5)
            LTD = row.get('Non_Current_Liabilities', D*0.5)
            if pd.isna(STD): STD = D*0.5
            if pd.isna(LTD): LTD = D*0.5
            
            def equations(p):
                V, sigma_V = p
                d1 = (np.log(V/D) + (r + 0.5*sigma_V**2)*T) / (sigma_V*np.sqrt(T))
                d2 = d1 - sigma_V*np.sqrt(T)
                return (V*norm.cdf(d1) - D*np.exp(-r*T)*norm.cdf(d2) - E,
                        V*norm.cdf(d1)*sigma_V - E*sigma_E)
            
            V, sigma_V = fsolve(equations, (E+D, sigma_E*0.5))
            DP = STD + 0.5*LTD
            # 분모 0 방지
            if V * sigma_V == 0:
                DD = np.nan
            else:
                DD = (V - DP) / (V * sigma_V)
            dd_list.append(DD)
        except:
            dd_list.append(np.nan)
    df['F2_KMV_DD'] = dd_list

    # Altman Z
    X1 = safe_div((df['Current_Assets'] - df['Current_Liabilities']), df['Total_Assets'])
    X2 = safe_div(df['Retained_Earnings'], df['Total_Assets'])
    X3 = safe_div(df['Operating_Income'], df['Total_Assets'])
    X4 = safe_div(df['Market_Cap'], df['Total_Liabilities'])
    X5 = safe_div(df['Revenue'], df['Total_Assets'])
    df['F3_Z_Score'] = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5

    # Beneish M-Score
    eps = 1e-6
    dsri = (df['Receivables']/(df['Revenue']+eps)) / (df['prev_Receivables']/(df['prev_Revenue']+eps) + eps)
    gp = df['Gross_Profit'].fillna(df['Revenue'] - df['COGS'])
    prev_gp = df['prev_Gross_Profit'].fillna(df['prev_Revenue'] - df['prev_COGS'])
    gmi = (prev_gp/(df['prev_Revenue']+eps)) / (gp/(df['Revenue']+eps) + eps)
    aq = 1 - (df['Current_Assets'] + df['PPE']) / (df['Total_Assets']+eps)
    prev_aq = 1 - (df['prev_Current_Assets'] + df['prev_PPE']) / (df['prev_Total_Assets']+eps)
    aqi = aq / (prev_aq + eps)
    sgi = df['Revenue'] / (df['prev_Revenue'] + eps)
    dep_rate = df['Final_Dep'] / (df['PPE'] + df['Final_Dep'] + eps)
    prev_dep_rate = df['prev_Final_Dep'] / (df['prev_PPE'] + df['prev_Final_Dep'] + eps)
    depi = prev_dep_rate / (dep_rate + eps)
    sgai = (df['SG&A']/(df['Revenue']+eps)) / (df['prev_SG&A']/(df['prev_Revenue']+eps) + eps)
    lvgi = (df['Total_Liabilities']/(df['Total_Assets']+eps)) / (df['prev_Total_Liabilities']/(df['prev_Total_Assets']+eps) + eps)
    tata = (df['Net_Income'] - df['CFO']) / (df['Total_Assets'] + eps)
    df['F4_M_Score'] = -4.84 + 0.920*dsri + 0.528*gmi + 0.404*aqi + 0.892*sgi + 0.115*depi - 0.172*sgai + 4.679*tata - 0.327*lvgi
    
    return df

# ==============================================================================
# 4. 메인 실행 함수
# ==============================================================================
def main():
    # 1. 대상 기업 리스트 준비
    try:
        df_distressed_list = pd.read_csv('관리종목_상장폐지_업종포함.csv') 
        df_normal_list = pd.read_csv('master_table_cleaned(2).csv')
    except Exception as e:
        print(f"리스트 파일 로드 실패: {e}")
        return

    df_distressed_list['Target_Code'] = df_distressed_list['종목코드'].apply(lambda x: str(x).zfill(6))
    df_distressed_list['Company_Name'] = df_distressed_list['회사명']
    df_distressed_list['Status'] = 'Distressed'
    
    df_normal_list['Target_Code'] = df_normal_list['종목코드'].apply(lambda x: str(x).zfill(6))
    df_normal_list['Company_Name'] = df_normal_list['종목명']
    df_normal_list['Status'] = 'Normal'
    
    all_companies = pd.concat([
        df_distressed_list[['Target_Code', 'Company_Name', 'Status']], 
        df_normal_list[['Target_Code', 'Company_Name', 'Status']]
    ])

    # ------------------------------------------------------------------
    # 이미 처리된 기업 확인 (이어하기 로직)
    # ------------------------------------------------------------------
    processed_codes = []
    
    if os.path.exists(OUTPUT_FILE):
        try:
            df_existing = pd.read_csv(OUTPUT_FILE, usecols=['Company_Code'], encoding='utf-8-sig')
            processed_codes = df_existing['Company_Code'].astype(str).str.zfill(6).unique().tolist()
            print(f"📁 기존 파일 발견: 총 {len(processed_codes)}개 기업이 이미 처리되었습니다.")
        except Exception as e:
            print(f"⚠️ 기존 파일 읽기 오류 (새로 시작): {e}")

    target_companies = all_companies[~all_companies['Target_Code'].isin(processed_codes)]
    print(f"🚀 남은 처리 대상 기업 수: {len(target_companies)} / 전체 {len(all_companies)}개\n")
    
    temp_rows = [] 
    
    for idx, (i, row) in enumerate(target_companies.iterrows()):
        code = row['Target_Code']
        name = row['Company_Name']
        status = row['Status']
        
        # 파일 경로 설정
        if status == 'Distressed':
            file_name = f"{code}_재무제표_1999_2025.xlsx"
        else:
            file_name = f"{code}_재무제표_2000_2025.xlsx"
            if not os.path.exists(os.path.join(DATA_FOLDER, file_name)):
                file_name = f"{code}_{name}.xlsx"
        
        full_path = os.path.join(DATA_FOLDER, file_name)
        
        if not os.path.exists(full_path):
            # 파일조차 없으면 어쩔 수 없음
            # print(f"[{idx+1}] {name}: 파일 없음 (Skip)")
            continue
            
        print(f"[{idx+1}/{len(target_companies)}] {name} ({code}) 처리 중...", end="")
        
        try:
            # 1. 재무 데이터 로드 (모든 시트 뒤지기)
            df_fin = load_financial_data(full_path)
            
            if df_fin.empty:
                print(" -> 재무데이터 추출 실패 ❌ (파일은 있으나 내용 인식 불가)")
                continue
                
            # 2. 로컬 주가 변동성 로드
            df_vol = load_local_stock_volatility(full_path)
            if not df_vol.empty:
                df_fin = pd.merge(df_fin, df_vol, on='bsns_year', how='left')
            else:
                df_fin['Volatility'] = np.nan
                
            # 3. 시가총액 수집 (Pykrx)
            min_year = int(df_fin['bsns_year'].min())
            max_year = int(df_fin['bsns_year'].max())
            
            df_cap = get_pykrx_market_cap(code, min_year, max_year)
            
            if not df_cap.empty:
                df_fin = pd.merge(df_fin, df_cap, on='bsns_year', how='left')
            else:
                df_fin['Market_Cap'] = np.nan
                
            # 4. Feature 계산
            df_calculated = calculate_features(df_fin)
            
            # 메타데이터 추가
            df_calculated['Company_Code'] = code
            df_calculated['Company_Name'] = name
            df_calculated['Status'] = status
            
            temp_rows.append(df_calculated)
            print(" -> 성공 ✅")

        except Exception as e:
            print(f" -> 에러 발생 ❌ ({e})")
            continue
        
        # ------------------------------------------------------------------
        # 중간 저장 (5개 단위)
        # ------------------------------------------------------------------
        if len(temp_rows) > 0 and (idx + 1) % 5 == 0:
            df_batch = pd.concat(temp_rows, ignore_index=True)
            header_status = not os.path.exists(OUTPUT_FILE)
            try:
                df_batch.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig', mode='a', header=header_status)
                temp_rows = [] 
                print(" 💾 저장 완료")
            except PermissionError:
                print(" ⚠️ 파일 열려있음! 저장 실패 (다음 텀에 시도)")

    # 반복문 종료 후 남은 데이터 저장
    if temp_rows:
        df_final = pd.concat(temp_rows, ignore_index=True)
        header_status = not os.path.exists(OUTPUT_FILE)
        try:
            df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig', mode='a', header=header_status)
        except Exception as e:
            print(f" ❌ 마지막 저장 실패: {e}")
        
    print("\n🎉 모든 작업 완료!")

if __name__ == "__main__":
    main()

📁 기존 파일 발견: 총 2757개 기업이 이미 처리되었습니다.
🚀 남은 처리 대상 기업 수: 853 / 전체 3984개

[708/853] 맥쿼리인프라 (088980) 처리 중... -> 재무데이터 추출 실패 ❌ (파일은 있으나 내용 인식 불가)
[709/853] 서울보증보험 (031210) 처리 중... -> 성공 ✅
[710/853] 에임드바이오 (0009K0) 처리 중... -> 성공 ✅
 💾 저장 완료
[711/853] 대한조선 (439260) 처리 중... -> 성공 ✅
[712/853] KB발해인프라 (415640) 처리 중... -> 재무데이터 추출 실패 ❌ (파일은 있으나 내용 인식 불가)
[713/853] 이뮨온시아 (424870) 처리 중... -> 성공 ✅
[714/853] 프로티나 (468530) 처리 중... -> 성공 ✅
[715/853] 명인제약 (317450) 처리 중... -> 성공 ✅
 💾 저장 완료
[716/853] 노타 (486990) 처리 중... -> 성공 ✅
[717/853] 씨엠티엑스 (388210) 처리 중... -> 성공 ✅
[718/853] 한라캐스트 (125490) 처리 중... -> 성공 ✅
[719/853] 테라뷰 (950250) 처리 중... -> 재무데이터 추출 실패 ❌ (파일은 있으나 내용 인식 불가)
[720/853] 큐리오시스 (494120) 처리 중... -> 성공 ✅
 💾 저장 완료
[721/853] 삼양컴텍 (484590) 처리 중... -> 성공 ✅
[722/853] 아이티켐 (309710) 처리 중... -> 성공 ✅
[723/853] 맵스리얼티 (094800) 처리 중... -> 재무데이터 추출 실패 ❌ (파일은 있으나 내용 인식 불가)
[724/853] 한텍 (098070) 처리 중... -> 성공 ✅
[725/853] 에스투더블유 (488280) 처리 중... -> 성공 ✅
 💾 저장 완료
[726/853] 한국피아이엠 (448900) 처리 중... -> 성공 ✅
[727/853] 

In [7]:
import pandas as pd
df = pd.read_csv('Bankruptcy_Prediction_Features_Final_V2.csv')
df["F1_Debt_Ratio"]

0        1.0
1        1.0
2        1.0
3        1.0
4        1.0
        ... 
22539    1.0
22540    1.0
22541    1.0
22542    1.0
22543    1.0
Name: F1_Debt_Ratio, Length: 22544, dtype: float64

최후의 수색..!

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve
from pykrx import stock
import time
import os
import warnings
import re

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 설정 및 계정과목 매핑
# ==============================================================================

DATA_FOLDER = "./" 
OUTPUT_FILE = 'Bankruptcy_Prediction_Features_Final_V2.csv' 

ACCOUNT_RULES = [
    ('Total_Assets', '자산총계', 'BS'), ('Total_Assets', '총자산', 'BS'),
    ('Total_Liabilities', '부채총계', 'BS'), ('Total_Liabilities', '총부채', 'BS'),
    ('Total_Equity', '자본총계', 'BS'), ('Total_Equity', '총자본', 'BS'),
    ('Current_Assets', '유동자산', 'BS'), ('Current_Liabilities', '유동부채', 'BS'),
    ('Non_Current_Liabilities', '비유동부채', 'BS'), ('Non_Current_Liabilities', '고정부채', 'BS'),
    ('Receivables', '매출채권', 'BS'), 
    ('Inventory', '재고자산', 'BS'), ('Inventory', '재고', 'BS'), 
    ('PPE', '유형자산', 'BS'),
    ('Accumulated_Depreciation', '감가상각누계액', 'BS'), ('Accumulated_Depreciation', '감가상각누계', 'BS'),
    ('Revenue', '매출액', 'IS'), ('Revenue', '영업수익', 'IS'), ('Revenue', '수익(매출액)', 'IS'),
    ('COGS', '매출원가', 'IS'), ('COGS', '영업비용', 'IS'), 
    ('Gross_Profit', '매출총이익', 'IS'),
    ('Operating_Income', '영업이익', 'IS'), ('Operating_Income', '영업손실', 'IS'),
    ('Net_Income', '당기순이익', 'IS'), ('Net_Income', '당기순손실', 'IS'),
    ('SG&A', '판매비와관리비', 'IS'), ('SG&A', '판관비', 'IS'),
    ('Interest_Expense', '이자비용', 'IS'), ('Interest_Expense', '금융비용', 'IS'),
    ('CFO', '영업활동', 'CF'), 
    ('Depreciation_CF', '감가상각비', 'CF'), 
    ('Retained_Earnings', '이익잉여금', 'BS'), ('Retained_Earnings', '결손금', 'BS'),
    ('Retained_Earnings', '미처분이익잉여금', 'BS'), ('Retained_Earnings', '미처리결손금', 'BS')
]

# ==============================================================================
# 2. 파일 찾기 및 데이터 로드 (검색 강화)
# ==============================================================================

def find_file_aggressive(folder, code, name):
    """
    [핵심 수정] 파일을 찾는 기준을 매우 완화하여 다시 찾습니다.
    1순위: 파일명이 종목코드로 시작
    2순위: 파일명 중간에 종목코드가 포함됨
    """
    try:
        files = os.listdir(folder)
        code = str(code).zfill(6)
        
        # 1. 코드로 시작하는 파일
        for f in files:
            if f.startswith(code) and (f.endswith('.xlsx') or f.endswith('.xls')):
                return os.path.join(folder, f)
        
        # 2. 파일명에 코드가 포함된 파일 (중간에 섞여 있는 경우)
        for f in files:
            if code in f and (f.endswith('.xlsx') or f.endswith('.xls')):
                return os.path.join(folder, f)
                
        return None
    except Exception as e:
        return None

def normalize_columns(df):
    df.columns = [str(c).strip().lower() for c in df.columns]
    rename_map = {
        '계정명': 'account_nm', '항목명': 'account_nm', 'account_name': 'account_nm', 'account': 'account_nm', 'item': 'account_nm',
        '당기': 'thstrm_amount', 'thstrm_amount': 'thstrm_amount', 'amount': 'thstrm_amount', '금액': 'thstrm_amount',
        '사업연도': 'bsns_year', 'year': 'bsns_year',
        '구분': 'sj_div', 'sj_div': 'sj_div'
    }
    df.rename(columns=rename_map, inplace=True)
    return df

def preprocess_wide_format(df):
    year_cols = [c for c in df.columns if re.match(r'^\d{4}$', str(c))]
    if len(year_cols) >= 1 and 'account_nm' in df.columns:
        try:
            df_melted = df.melt(id_vars=['account_nm', 'sj_div'] if 'sj_div' in df.columns else ['account_nm'], 
                                value_vars=year_cols, 
                                var_name='bsns_year', 
                                value_name='thstrm_amount')
            return df_melted
        except:
            return df
    return df

def find_header_and_load(df_raw):
    sample_cols = [str(c).lower() for c in df_raw.columns]
    keywords = ['계정명', 'account', '항목명', 'account_nm', 'item', '자산총계', '매출액']
    if any(k in str(sample_cols) for k in keywords): return df_raw

    for i in range(min(30, len(df_raw))):
        row_values = df_raw.iloc[i].astype(str).tolist()
        if any(k in str(row_values).lower() for k in keywords):
            new_header = df_raw.iloc[i]
            df_new = df_raw[i+1:].copy()
            df_new.columns = new_header
            return df_new
    return df_raw

def pivot_data(df):
    if df.empty: return pd.DataFrame()
    df = find_header_and_load(df)
    df = normalize_columns(df)
    df = preprocess_wide_format(df)
    
    if 'account_nm' not in df.columns: return pd.DataFrame()
    if 'sj_div' not in df.columns: df['sj_div'] = 'ALL'
    
    df['Std_Account'] = np.nan
    for std_acc, keyword, div_cond in ACCOUNT_RULES:
        cond_keyword = df['account_nm'].astype(str).str.contains(keyword, na=False)
        if div_cond == 'CF':
            cond_div = df['sj_div'].astype(str).str.contains('CF', case=False) | df['sj_div'].astype(str).str.contains('현금', na=False)
        elif div_cond == 'BS':
            cond_div = df['sj_div'].isin(['BS', '재무상태표'])
        elif div_cond == 'IS':
            cond_div = df['sj_div'].isin(['IS', '손익계산서', '포괄손익계산서', 'CIS'])
        else:
            cond_div = True 
        mask = cond_keyword & cond_div
        df.loc[mask, 'Std_Account'] = std_acc
    
    df_clean = df.dropna(subset=['Std_Account'])
    if 'thstrm_amount' in df_clean.columns:
        df_clean['thstrm_amount'] = (df_clean['thstrm_amount'].astype(str).str.replace(',', '').str.replace('(', '-').str.replace(')', '').apply(pd.to_numeric, errors='coerce'))
    else: return pd.DataFrame()
    
    if 'bsns_year' not in df_clean.columns: return pd.DataFrame()
    df_clean['bsns_year'] = pd.to_numeric(df_clean['bsns_year'], errors='coerce')
    df_clean = df_clean.dropna(subset=['bsns_year'])
    
    return df_clean.pivot_table(index='bsns_year', columns='Std_Account', values='thstrm_amount', aggfunc='first')

def load_financial_data(filepath):
    try:
        xls = pd.ExcelFile(filepath)
        df_fin = pd.DataFrame()
        for sheet in xls.sheet_names:
            try:
                df_temp = pd.read_excel(xls, sheet_name=sheet)
                df_pivot = pivot_data(df_temp)
                if not df_pivot.empty:
                    if 'Total_Assets' in df_pivot.columns or 'Revenue' in df_pivot.columns:
                        df_fin = df_pivot
                        break
            except: continue
        if df_fin.empty: return pd.DataFrame()
        return df_fin.reset_index()
    except: return pd.DataFrame()

def load_local_stock_volatility(filepath):
    try:
        if filepath.endswith('.xlsx'):
            try: df_price = pd.read_excel(filepath, sheet_name='Stock_Price')
            except: return pd.DataFrame()
        else: df_price = pd.read_csv(filepath)
        if df_price.empty: return pd.DataFrame()
        
        date_col = next((c for c in df_price.columns if 'date' in str(c).lower() or '일자' in str(c)), None)
        close_col = next((c for c in df_price.columns if 'close' in str(c).lower() or '종가' in str(c)), None)
        if not date_col or not close_col: return pd.DataFrame()
        
        df_price[date_col] = pd.to_datetime(df_price[date_col], errors='coerce')
        df_price['Year'] = df_price[date_col].dt.year
        df_price[close_col] = pd.to_numeric(df_price[close_col], errors='coerce')
        df_price['Return'] = df_price.groupby('Year')[close_col].pct_change()
        volatility = df_price.groupby('Year')['Return'].std() * np.sqrt(252)
        return volatility.reset_index().rename(columns={'Year': 'bsns_year', 'Return': 'Volatility'})
    except: return pd.DataFrame()

def get_pykrx_market_cap(ticker, start_year, end_year):
    results = []
    ticker = str(ticker).zfill(6)
    if start_year < 2000: start_year = 2000
    if end_year > 2025: end_year = 2025
    for year in range(start_year, end_year + 1):
        try:
            ohlcv = stock.get_market_ohlcv(str(year)+"1215", str(year)+"1231", ticker)
            if ohlcv is None or ohlcv.empty: continue
            last_date = ohlcv.index[-1]
            last_date_str = last_date.strftime("%Y%m%d")
            cap_df = stock.get_market_cap(last_date_str, last_date_str, ticker)
            if not cap_df.empty:
                mc = cap_df['시가총액'].iloc[0] if '시가총액' in cap_df.columns else np.nan
                results.append({'bsns_year': year, 'Market_Cap': mc})
            time.sleep(0.1)
        except: continue
    return pd.DataFrame(results)

def ensure_columns(df):
    required = set([rule[0] for rule in ACCOUNT_RULES])
    for col in required:
        if col not in df.columns: df[col] = np.nan
    return df

# ==============================================================================
# 3. Feature 계산 함수
# ==============================================================================
def calculate_features(df):
    df = ensure_columns(df)
    df = df.copy()
    df = df.sort_values(by='bsns_year')
    
    cols = [c for c in df.columns if c not in ['bsns_year', 'Market_Cap', 'Volatility']]
    for c in cols: df[f'prev_{c}'] = df[c].shift(1)

    df['Dep_Proxy'] = df['Accumulated_Depreciation'] - df['prev_Accumulated_Depreciation']
    df['Dep_Proxy'] = df['Dep_Proxy'].apply(lambda x: x if (pd.notnull(x) and x > 0) else 0)
    df['Final_Dep'] = df['Depreciation_CF'].fillna(df['Dep_Proxy']).fillna(0)
    df['prev_Final_Dep'] = df['Final_Dep'].shift(1)

    def safe_div(a, b): return np.where((b == 0) | pd.isna(b), np.nan, a / b)

    df['F1_Equity_Growth'] = safe_div((df['Total_Equity'] - df['prev_Total_Equity']), df['prev_Total_Equity'].abs())
    df['F1_Retained_Earnings_Ratio'] = safe_div(df['Retained_Earnings'], df['Total_Assets'])
    df['F1_ROA'] = safe_div(df['Net_Income'], df['Total_Assets'])
    df['F1_Debt_Ratio'] = safe_div(df['Total_Liabilities'], df['Total_Assets'])
    df['F1_Current_Ratio'] = safe_div(df['Current_Assets'], df['Current_Liabilities'])
    df['F1_ROE'] = safe_div(df['Net_Income'], df['Total_Equity'])
    df['F1_Interest_Coverage'] = safe_div(df['Operating_Income'], df['Interest_Expense'])

    r, T = 0.035, 1
    df['Vol_Adj'] = df['Volatility'].fillna(method='ffill').fillna(0.6)
    
    dd_list = []
    for idx, row in df.iterrows():
        try:
            E, sigma_E, D = row['Market_Cap'], row['Vol_Adj'], row['Total_Liabilities']
            if pd.isna(E) or pd.isna(D) or E <= 0 or D < 0:
                dd_list.append(np.nan); continue
            
            STD = row.get('Current_Liabilities', D*0.5)
            LTD = row.get('Non_Current_Liabilities', D*0.5)
            if pd.isna(STD): STD = D*0.5
            if pd.isna(LTD): LTD = D*0.5
            
            def equations(p):
                V, sigma_V = p
                d1 = (np.log(V/D) + (r + 0.5*sigma_V**2)*T) / (sigma_V*np.sqrt(T))
                d2 = d1 - sigma_V*np.sqrt(T)
                return (V*norm.cdf(d1) - D*np.exp(-r*T)*norm.cdf(d2) - E, V*norm.cdf(d1)*sigma_V - E*sigma_E)
            
            V, sigma_V = fsolve(equations, (E+D, sigma_E*0.5))
            DP = STD + 0.5*LTD
            dd_list.append((V - DP) / (V * sigma_V) if V * sigma_V != 0 else np.nan)
        except: dd_list.append(np.nan)
    df['F2_KMV_DD'] = dd_list

    X1 = safe_div((df['Current_Assets'] - df['Current_Liabilities']), df['Total_Assets'])
    X2 = safe_div(df['Retained_Earnings'], df['Total_Assets'])
    X3 = safe_div(df['Operating_Income'], df['Total_Assets'])
    X4 = safe_div(df['Market_Cap'], df['Total_Liabilities'])
    X5 = safe_div(df['Revenue'], df['Total_Assets'])
    df['F3_Z_Score'] = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5

    eps = 1e-6
    dsri = safe_div(safe_div(df['Receivables'], df['Revenue']), safe_div(df['prev_Receivables'], df['prev_Revenue']))
    gp = df['Gross_Profit'].fillna(df['Revenue'] - df['COGS'])
    prev_gp = df['prev_Gross_Profit'].fillna(df['prev_Revenue'] - df['prev_COGS'])
    gmi = safe_div(safe_div(prev_gp, df['prev_Revenue']), safe_div(gp, df['Revenue']))
    aq = 1 - safe_div(df['Current_Assets'] + df['PPE'], df['Total_Assets'])
    prev_aq = 1 - safe_div(df['prev_Current_Assets'] + df['prev_PPE'], df['prev_Total_Assets'])
    aqi = safe_div(aq, prev_aq)
    sgi = safe_div(df['Revenue'], df['prev_Revenue'])
    dep_rate = safe_div(df['Final_Dep'], df['PPE'] + df['Final_Dep'])
    prev_dep_rate = safe_div(df['prev_Final_Dep'], df['prev_PPE'] + df['prev_Final_Dep'])
    depi = safe_div(prev_dep_rate, dep_rate)
    sgai = safe_div(safe_div(df['SG&A'], df['Revenue']), safe_div(df['prev_SG&A'], df['prev_Revenue']))
    lvgi = safe_div(safe_div(df['Total_Liabilities'], df['Total_Assets']), safe_div(df['prev_Total_Liabilities'], df['prev_Total_Assets']))
    tata = safe_div(df['Net_Income'] - df['CFO'], df['Total_Assets'])
    df['F4_M_Score'] = -4.84 + 0.920*dsri + 0.528*gmi + 0.404*aqi + 0.892*sgi + 0.115*depi - 0.172*sgai + 4.679*tata - 0.327*lvgi
    return df

# ==============================================================================
# 4. 메인 실행 함수 (마무리 수색)
# ==============================================================================
def main():
    try:
        df_distressed = pd.read_csv('관리종목_상장폐지_업종포함.csv') 
        df_normal = pd.read_csv('master_table_cleaned(2).csv')
    except Exception as e:
        print(f"리스트 로드 실패: {e}"); return

    df_distressed['Target_Code'] = df_distressed['종목코드'].apply(lambda x: str(x).zfill(6))
    df_distressed['Company_Name'] = df_distressed['회사명']
    df_distressed['Status'] = 'Distressed'
    df_normal['Target_Code'] = df_normal['종목코드'].apply(lambda x: str(x).zfill(6))
    df_normal['Company_Name'] = df_normal['종목명']
    df_normal['Status'] = 'Normal'
    
    all_companies = pd.concat([df_distressed[['Target_Code', 'Company_Name', 'Status']], df_normal[['Target_Code', 'Company_Name', 'Status']]])

    processed_codes = []
    if os.path.exists(OUTPUT_FILE):
        try:
            df_existing = pd.read_csv(OUTPUT_FILE, usecols=['Company_Code'], encoding='utf-8-sig')
            processed_codes = df_existing['Company_Code'].astype(str).str.zfill(6).unique().tolist()
            print(f"📁 기존 작업 확인: {len(processed_codes)}개 기업 완료됨.")
        except: pass

    # 남은 기업들 필터링
    target_companies = all_companies[~all_companies['Target_Code'].isin(processed_codes)]
    print(f"🚀 최종 처리 대상(Missing List): {len(target_companies)}개\n")
    
    temp_rows = [] 
    
    for idx, (i, row) in enumerate(target_companies.iterrows()):
        code = row['Target_Code']
        name = row['Company_Name']
        status = row['Status']
        
        # [수정] 파일명 정밀 탐색 (코드 포함 여부 확인)
        full_path = find_file_aggressive(DATA_FOLDER, code, name)
        
        if not full_path:
            # 여기는 print를 안하거나, 마지막으로 "진짜 없음" 로그를 남겨도 됨
            print(f"[{idx+1}] {name}({code}): ❌ 파일 절대 없음 (Skip)")
            continue
            
        print(f"[{idx+1}/{len(target_companies)}] {name} ({code}) 🔎 파일 발견 -> 분석 시도...", end="")
        
        try:
            df_fin = load_financial_data(full_path)
            
            if df_fin.empty:
                print(" -> ❌ 데이터 없음/형식 불일치")
                continue
                
            df_vol = load_local_stock_volatility(full_path)
            if not df_vol.empty: df_fin = pd.merge(df_fin, df_vol, on='bsns_year', how='left')
            else: df_fin['Volatility'] = np.nan
            
            min_year = int(df_fin['bsns_year'].min())
            max_year = int(df_fin['bsns_year'].max())
            df_cap = get_pykrx_market_cap(code, min_year, max_year)
            
            if not df_cap.empty: df_fin = pd.merge(df_fin, df_cap, on='bsns_year', how='left')
            else: df_fin['Market_Cap'] = np.nan
                
            df_calculated = calculate_features(df_fin)
            df_calculated['Company_Code'] = code
            df_calculated['Company_Name'] = name
            df_calculated['Status'] = status
            
            temp_rows.append(df_calculated)
            print(" -> ✅ 성공")

        except Exception as e:
            print(f" -> ❌ 에러: {e}")
            continue
        
        # 중간 저장 (5개 단위)
        if len(temp_rows) > 0 and (idx + 1) % 5 == 0:
            df_batch = pd.concat(temp_rows, ignore_index=True)
            header_status = not os.path.exists(OUTPUT_FILE)
            try:
                df_batch.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig', mode='a', header=header_status)
                temp_rows = [] 
                print("   💾 저장됨")
            except: print("   ⚠️ 저장 실패 (파일 열려있음)")

    if temp_rows:
        df_final = pd.concat(temp_rows, ignore_index=True)
        header_status = not os.path.exists(OUTPUT_FILE)
        try: df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig', mode='a', header=header_status)
        except: pass
        
    print("\n🎉 최종 작업 종료! 이래도 없으면 진짜 없는 것입니다.")

if __name__ == "__main__":
    main()

📁 기존 작업 확인: 2803개 기업 완료됨.
🚀 최종 처리 대상(Missing List): 807개

[1] 디아이디(074130): ❌ 파일 절대 없음 (Skip)
[2] 엔케이바이오(019260): ❌ 파일 절대 없음 (Skip)
[3] AD모터스(038120): ❌ 파일 절대 없음 (Skip)
[4] 아큐텍(013780): ❌ 파일 절대 없음 (Skip)
[5] 유에이블(071530): ❌ 파일 절대 없음 (Skip)
[6] 이룸지엔지(050640): ❌ 파일 절대 없음 (Skip)
[7] 더체인지(054120): ❌ 파일 절대 없음 (Skip)
[8] 히스토스템(036840): ❌ 파일 절대 없음 (Skip)
[9] 이앤텍(047450): ❌ 파일 절대 없음 (Skip)
[10] 네이쳐글로벌(088020): ❌ 파일 절대 없음 (Skip)
[11] 태광이엔시(048140): ❌ 파일 절대 없음 (Skip)
[12] 쎄라텍(041550): ❌ 파일 절대 없음 (Skip)
[13] 액티투오(047710): ❌ 파일 절대 없음 (Skip)
[14] 선우중공업(068770): ❌ 파일 절대 없음 (Skip)
[15] 디에이치패션(045260): ❌ 파일 절대 없음 (Skip)
[16] 바이나믹(067850): ❌ 파일 절대 없음 (Skip)
[17] 아이알디(084810): ❌ 파일 절대 없음 (Skip)
[18] 지오엠씨(033030): ❌ 파일 절대 없음 (Skip)
[19] 코디콤(041800): ❌ 파일 절대 없음 (Skip)
[20] 코어비트(056850): ❌ 파일 절대 없음 (Skip)
[21] 티이씨(067950): ❌ 파일 절대 없음 (Skip)
[22] 제너비오믹스(017010): ❌ 파일 절대 없음 (Skip)
[23] 올리브나인(052970): ❌ 파일 절대 없음 (Skip)
[24] 하이럭스(079340): ❌ 파일 절대 없음 (Skip)
[25] 스타맥스(017050): ❌ 파일 절대 없음 (Skip)
[26] 알에스넷(046430)

In [5]:
df_final.head()

NameError: name 'df_final' is not defined

In [8]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 설정
# ==============================================================================
input_file = 'Bankruptcy_Prediction_Features_Final_V2.csv'
output_file = 'Bankruptcy_Prediction_Features_Final_V2_Fixed.csv' # 수정된 파일명

# ==============================================================================
# 데이터 로드 및 수정
# ==============================================================================
if os.path.exists(input_file):
    print("📂 데이터 로드 중...")
    df = pd.read_csv(input_file)
    
    # 1. Total_Liabilities 재계산 (유동부채 + 비유동부채)
    # NaN 값은 0으로 처리하여 합산 (둘 중 하나만 있어도 계산되도록)
    if 'Current_Liabilities' in df.columns and 'Non_Current_Liabilities' in df.columns:
        df['Total_Liabilities'] = df['Current_Liabilities'].fillna(0) + df['Non_Current_Liabilities'].fillna(0)
        print("✅ Total_Liabilities 수정 완료")
    else:
        print("⚠️ 경고: 유동부채 또는 비유동부채 컬럼이 없어 Total_Liabilities를 수정하지 못했습니다.")

    # 2. prev_Total_Liabilities 재계산 (전기 유동부채 + 전기 비유동부채)
    if 'prev_Current_Liabilities' in df.columns and 'prev_Non_Current_Liabilities' in df.columns:
        df['prev_Total_Liabilities'] = df['prev_Current_Liabilities'].fillna(0) + df['prev_Non_Current_Liabilities'].fillna(0)
        print("✅ prev_Total_Liabilities 수정 완료")
    else:
        # 만약 prev 컬럼이 없다면, 방금 수정한 Total_Liabilities를 shift해서 계산
        print("ℹ️ prev 컬럼이 없어 shift 연산으로 대체합니다.")
        df = df.sort_values(by=['Company_Code', 'bsns_year'])
        df['prev_Total_Liabilities'] = df.groupby('Company_Code')['Total_Liabilities'].shift(1)
        print("✅ prev_Total_Liabilities 수정 완료 (Shift 방식)")

    # 3. F1_Debt_Ratio 재계산 (수정된 총부채 / 총자산)
    if 'Total_Assets' in df.columns:
        # 0으로 나누기 방지
        df['F1_Debt_Ratio'] = np.where(
            (df['Total_Assets'] == 0) | (df['Total_Assets'].isna()), 
            np.nan, 
            df['Total_Liabilities'] / df['Total_Assets']
        )
        print("✅ F1_Debt_Ratio 수정 완료")
    
    # ==============================================================================
    # 저장
    # ==============================================================================
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n🎉 모든 수정이 완료되었습니다! '{output_file}' 파일을 확인하세요.")
    
    # 결과 미리보기 (상위 5개)
    print("\n[수정된 데이터 미리보기]")
    print(df[['Total_Assets', 'Total_Liabilities', 'Current_Liabilities', 'Non_Current_Liabilities', 'F1_Debt_Ratio']].head())

else:
    print(f"❌ 오류: '{input_file}' 파일을 찾을 수 없습니다.")

📂 데이터 로드 중...
✅ Total_Liabilities 수정 완료
✅ prev_Total_Liabilities 수정 완료
✅ F1_Debt_Ratio 수정 완료

🎉 모든 수정이 완료되었습니다! 'Bankruptcy_Prediction_Features_Final_V2_Fixed.csv' 파일을 확인하세요.

[수정된 데이터 미리보기]
   Total_Assets  Total_Liabilities  Current_Liabilities  \
0  8.267376e+10       2.081225e+10         1.982200e+10   
1  9.052316e+10       2.099571e+10         1.990349e+10   
2  9.029187e+10       3.500764e+09         3.455230e+09   
3  9.263018e+10       4.747022e+09         4.745904e+09   
4  1.067517e+11       1.010004e+10         7.271190e+09   

   Non_Current_Liabilities  F1_Debt_Ratio  
0             9.902483e+08       0.251739  
1             1.092222e+09       0.231937  
2             4.553450e+07       0.038772  
3             1.118100e+06       0.051247  
4             2.828855e+09       0.094613  


In [9]:
df.head()

,bsns_year,CFO,COGS,Current_Assets,Current_Liabilities,Depreciation_CF,Gross_Profit,Inventory,Net_Income,Non_Current_Liabilities,...,F1_Current_Ratio,F1_ROE,F1_Interest_Coverage,Vol_Adj,F2_KMV_DD,F3_Z_Score,F4_M_Score,Company_Code,Company_Name,Status
0,2015,6.885299e+09,2.629576e+10,5.932923e+10,1.982200e+10,1.048120e+09,3.549896e+10,6.730109e+09,5.635160e+09,9.902483e+08,...,2.993100,0.091093,NaN,0.577378,2.181834,3.509617,NaN,071200,인피니트헬스케어,Distressed
1,2016,5.744385e+09,3.193205e+10,6.601013e+10,1.990349e+10,9.023429e+08,3.956563e+10,4.415341e+09,7.174766e+09,1.092222e+09,...,3.316511,0.103193,NaN,0.334743,4.182605,3.183807,-1.008438,071200,인피니트헬스케어,Distressed
2,2017,-2.134783e+09,3.228929e+10,9.589554e+08,3.455230e+09,5.431895e+08,4.048486e+10,1.155644e+09,1.495849e+09,4.553450e+07,...,0.277537,0.021194,NaN,0.475473,2.978670,2.767874,-1.150158,071200,인피니트헬스케어,Distressed
3,2018,-1.677706e+09,2.482455e+10,1.208131e+09,4.745904e+09,9.497713e+08,3.949078e+10,1.357180e+09,4.837012e+09,1.118100e+06,...,0.254563,0.062862,NaN,0.512478,3.097087,2.183762,-1.316594,071200,인피니트헬스케어,Distressed
4,2019,1.644548e+09,2.744862e+10,1.479918e+09,7.271190e+09,1.725876e+09,4.681841e+10,2.006279e+09,7.268999e+09,2.828855e+09,...,0.203532,0.086217,NaN,0.304345,5.489066,2.185795,-2.328118,071200,인피니트헬스케어,Distressed


In [10]:
import pandas as pd

# 1. 데이터 로드 (수정된 파일 사용 권장)
file_path = 'Bankruptcy_Prediction_Features_Final_V2_Fixed.csv'
df = pd.read_csv(file_path)

# 2. 남길 칼럼 정의
# (1) 식별자 (ID)
id_cols = ['bsns_year', 'Company_Code', 'Company_Name', 'Status']

# (2) 학습에 사용할 Feature 선택
# 방법 A: 'F'로 시작하는 파생변수 + Vol_Adj(변동성) 등 모델링 핵심 변수 자동 선택
# (만약 원본 재무데이터도 넣고 싶다면 이 리스트에 추가하면 됩니다)
feature_cols = [col for col in df.columns if col.startswith('F') or col in ['Vol_Adj', 'Market_Cap']]

# 3. 최종 데이터셋 생성
final_cols = id_cols + feature_cols
df_model = df[final_cols].copy()

# 4. (선택사항) Status를 숫자로 변환 (Normal=0, Distressed=1)
# 머신러닝 모델은 문자열을 이해하지 못하므로 변환이 필요합니다.
status_mapping = {'Normal': 0, 'Distressed': 1}
df_model['Target'] = df_model['Status'].map(status_mapping)

# 5. 결과 확인
print(f"선택된 칼럼 개수: {len(df_model.columns)}")
print(f"선택된 Feature 목록: {feature_cols}")
print("-" * 50)
df_model.head()

# 6. 모델링용 파일 별도 저장
df_model.to_csv('Bankruptcy_Dataset_For_Modeling.csv', index=False, encoding='utf-8-sig')

선택된 칼럼 개수: 18
선택된 Feature 목록: ['Market_Cap', 'Final_Dep', 'F1_Equity_Growth', 'F1_Retained_Earnings_Ratio', 'F1_ROA', 'F1_Debt_Ratio', 'F1_Current_Ratio', 'F1_ROE', 'F1_Interest_Coverage', 'Vol_Adj', 'F2_KMV_DD', 'F3_Z_Score', 'F4_M_Score']
--------------------------------------------------


In [11]:
df_model

,bsns_year,Company_Code,Company_Name,Status,Market_Cap,Final_Dep,F1_Equity_Growth,F1_Retained_Earnings_Ratio,F1_ROA,F1_Debt_Ratio,F1_Current_Ratio,F1_ROE,F1_Interest_Coverage,Vol_Adj,F2_KMV_DD,F3_Z_Score,F4_M_Score,Target
0,2015,071200,인피니트헬스케어,Distressed,2.286523e+11,1.048120e+09,NaN,0.251549,0.068161,0.251739,2.993100,0.091093,NaN,0.577378,2.181834,3.509617,NaN,1
1,2016,071200,인피니트헬스케어,Distressed,1.673597e+11,9.023429e+08,0.123921,0.308238,0.079259,0.231937,3.316511,0.103193,NaN,0.334743,4.182605,3.183807,-1.008438,1
2,2017,071200,인피니트헬스케어,Distressed,2.010268e+11,5.431895e+08,0.015111,0.326141,0.016567,0.038772,0.277537,0.021194,NaN,0.475473,2.978670,2.767874,-1.150158,1
3,2018,071200,인피니트헬스케어,Distressed,1.439391e+11,9.497713e+08,0.090232,0.370814,0.052219,0.051247,0.254563,0.062862,NaN,0.512478,3.097087,2.183762,-1.316594,1
4,2019,071200,인피니트헬스케어,Distressed,1.407676e+11,1.725876e+09,0.095699,0.389559,0.068093,0.094613,0.203532,0.086217,NaN,0.304345,5.489066,2.185795,-2.328118,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22539,2025,364950,에이아이코리아,Normal,6.969014e+10,0.000000e+00,NaN,0.244639,-0.008910,0.385823,2.143624,-0.014508,NaN,0.753522,2.237538,1.721080,NaN,0
22540,2025,455180,케이지에이,Normal,NaN,0.000000e+00,NaN,-0.029277,-0.056809,0.307518,1.658611,-0.126379,NaN,0.696845,3.071875,0.444168,NaN,0
22541,2025,342870,오아,Normal,NaN,0.000000e+00,NaN,0.218109,NaN,1.260477,1.185227,NaN,8.234492,0.402903,4.030544,1.326381,NaN,0
22542,2025,462310,뉴키즈온,Normal,4.693788e+10,3.568058e+08,NaN,0.180441,NaN,0.205047,4.207594,NaN,2.952312,0.650843,2.916456,1.665656,NaN,0


In [12]:
# 제외할 칼럼 목록 정의
drop_cols = ['Market_Cap', 'Final_Dep', 'Vol_Adj']

# 해당 칼럼들 제거 (errors='ignore'를 넣으면 혹시 칼럼이 없어도 에러가 안 납니다)
df_model = df_model.drop(columns=drop_cols, errors='ignore')

# 결과 확인
print("최종 남은 칼럼 목록:")
print(df_model.columns.tolist())

최종 남은 칼럼 목록:
['bsns_year', 'Company_Code', 'Company_Name', 'Status', 'F1_Equity_Growth', 'F1_Retained_Earnings_Ratio', 'F1_ROA', 'F1_Debt_Ratio', 'F1_Current_Ratio', 'F1_ROE', 'F1_Interest_Coverage', 'F2_KMV_DD', 'F3_Z_Score', 'F4_M_Score', 'Target']


In [13]:
df_model

,bsns_year,Company_Code,Company_Name,Status,F1_Equity_Growth,F1_Retained_Earnings_Ratio,F1_ROA,F1_Debt_Ratio,F1_Current_Ratio,F1_ROE,F1_Interest_Coverage,F2_KMV_DD,F3_Z_Score,F4_M_Score,Target
0,2015,071200,인피니트헬스케어,Distressed,NaN,0.251549,0.068161,0.251739,2.993100,0.091093,NaN,2.181834,3.509617,NaN,1
1,2016,071200,인피니트헬스케어,Distressed,0.123921,0.308238,0.079259,0.231937,3.316511,0.103193,NaN,4.182605,3.183807,-1.008438,1
2,2017,071200,인피니트헬스케어,Distressed,0.015111,0.326141,0.016567,0.038772,0.277537,0.021194,NaN,2.978670,2.767874,-1.150158,1
3,2018,071200,인피니트헬스케어,Distressed,0.090232,0.370814,0.052219,0.051247,0.254563,0.062862,NaN,3.097087,2.183762,-1.316594,1
4,2019,071200,인피니트헬스케어,Distressed,0.095699,0.389559,0.068093,0.094613,0.203532,0.086217,NaN,5.489066,2.185795,-2.328118,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22539,2025,364950,에이아이코리아,Normal,NaN,0.244639,-0.008910,0.385823,2.143624,-0.014508,NaN,2.237538,1.721080,NaN,0
22540,2025,455180,케이지에이,Normal,NaN,-0.029277,-0.056809,0.307518,1.658611,-0.126379,NaN,3.071875,0.444168,NaN,0
22541,2025,342870,오아,Normal,NaN,0.218109,NaN,1.260477,1.185227,NaN,8.234492,4.030544,1.326381,NaN,0
22542,2025,462310,뉴키즈온,Normal,NaN,0.180441,NaN,0.205047,4.207594,NaN,2.952312,2.916456,1.665656,NaN,0


In [15]:
import pandas as pd
import numpy as np

# ==============================================================================
# 1. 거시경제 지표 데이터 생성
# ==============================================================================
macro_data = {
    'bsns_year': [
        1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 
        2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 
        2019, 2020, 2021, 2022, 2023, 2024, 2025
    ],
    'M_Short_Term_Rate': [ # 단기금리
        4.75, 5.25, 4.00, 4.25, 3.75, 3.25, 3.75, 4.50, 5.00, 3.00,
        2.00, 2.50, 3.25, 2.75, 2.50, 2.00, 1.50, 1.25, 1.50, 1.75,
        1.25, 0.50, 1.00, 3.25, 3.50, 3.00, 2.50
    ],
    'M_Long_Term_Rate': [ # 장기금리
        8.84, 8.28, 7.05, 6.81, 5.42, 4.81, 4.70, 5.16, 5.17, 5.75,
        4.17, 4.26, 3.99, 3.42, 3.28, 3.04, 2.31, 1.75, 2.28, 2.47,
        1.62, 1.39, 1.83, 3.12, 3.65, 3.20, 2.90
    ],
    'M_Rate_Spread': [ # 장단기 금리차
        4.09, 3.03, 3.05, 2.56, 1.67, 1.56, 0.95, 0.66, 0.17, 2.75,
        2.17, 1.76, 0.74, 0.67, 0.78, 1.04, 0.81, 0.50, 0.78, 0.72,
        0.37, 0.89, 0.83, -0.13, 0.15, 0.20, 0.40
    ],
    'M_Nominal_GDP_Growth': [ # 명목 GDP 성장률
        11.2, 11.6, 8.7, 11.2, 7.1, 8.2, 5.7, 5.5, 7.2, 7.5,
        3.8, 10.3, 5.3, 3.4, 3.7, 3.7, 5.4, 4.8, 5.5, 3.4,
        1.4, 0.4, 6.8, 4.3, 3.6, 4.5, 3.5
    ],
    'M_Real_GDP_Growth': [ # 실질 GDP 성장률
        11.5, 9.1, 4.9, 7.7, 3.1, 5.2, 4.3, 5.3, 5.8, 3.0,
        0.8, 6.8, 3.7, 2.4, 3.2, 3.2, 2.8, 2.9, 3.2, 2.9,
        2.2, -0.7, 4.3, 2.6, 1.4, 2.2, 1.1
    ],
    'M_Inflation': [ # 인플레이션
        0.8, 2.3, 4.1, 2.8, 3.5, 3.6, 2.8, 2.2, 2.5, 4.7,
        2.8, 2.9, 4.0, 2.2, 1.3, 1.3, 0.7, 1.0, 1.9, 1.5,
        0.4, 0.5, 2.5, 5.1, 3.6, 2.4, 2.1
    ],
    'M_Exchange_Rate': [ # 환율 (콤마 제거 후 숫자형으로 입력)
        1189, 1131, 1291, 1251, 1192, 1145, 1024, 955, 929, 1102,
        1276, 1156, 1108, 1127, 1095, 1053, 1131, 1160, 1131, 1100,
        1166, 1180, 1144, 1292, 1305, 1360, 1450
    ]
}

df_macro = pd.DataFrame(macro_data)

# ==============================================================================
# 2. df_model에 병합 (Merge)
# ==============================================================================
# df_model의 bsns_year가 숫자형인지 확인하고 변환 (안전장치)
df_model['bsns_year'] = pd.to_numeric(df_model['bsns_year'], errors='coerce')

# 연도(bsns_year) 기준으로 Left Join
df_final = pd.merge(df_model, df_macro, on='bsns_year', how='left')

# ==============================================================================
# 3. 결과 확인 및 저장
# ==============================================================================
print("📊 거시경제 지표 병합 완료!")
print(f"기존 컬럼 수: {len(df_model.columns)} -> 병합 후 컬럼 수: {len(df_final.columns)}")
print("-" * 50)
print(df_final[['bsns_year', 'M_Short_Term_Rate', 'M_Real_GDP_Growth', 'M_Exchange_Rate']].head()) # 샘플 출력

# 최종 파일 저장
output_file = 'Bankruptcy_Dataset_For_Modeling_With_Macro.csv'
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✅ 최종 파일 저장됨: {output_file}")

📊 거시경제 지표 병합 완료!
기존 컬럼 수: 15 -> 병합 후 컬럼 수: 22
--------------------------------------------------
   bsns_year  M_Short_Term_Rate  M_Real_GDP_Growth  M_Exchange_Rate
0       2015               1.50                2.8             1131
1       2016               1.25                2.9             1160
2       2017               1.50                3.2             1131
3       2018               1.75                2.9             1100
4       2019               1.25                2.2             1166

✅ 최종 파일 저장됨: Bankruptcy_Dataset_For_Modeling_With_Macro.csv


In [16]:
df_final

,bsns_year,Company_Code,Company_Name,Status,F1_Equity_Growth,F1_Retained_Earnings_Ratio,F1_ROA,F1_Debt_Ratio,F1_Current_Ratio,F1_ROE,...,F3_Z_Score,F4_M_Score,Target,M_Short_Term_Rate,M_Long_Term_Rate,M_Rate_Spread,M_Nominal_GDP_Growth,M_Real_GDP_Growth,M_Inflation,M_Exchange_Rate
0,2015,071200,인피니트헬스케어,Distressed,NaN,0.251549,0.068161,0.251739,2.993100,0.091093,...,3.509617,NaN,1,1.50,2.31,0.81,5.4,2.8,0.7,1131
1,2016,071200,인피니트헬스케어,Distressed,0.123921,0.308238,0.079259,0.231937,3.316511,0.103193,...,3.183807,-1.008438,1,1.25,1.75,0.50,4.8,2.9,1.0,1160
2,2017,071200,인피니트헬스케어,Distressed,0.015111,0.326141,0.016567,0.038772,0.277537,0.021194,...,2.767874,-1.150158,1,1.50,2.28,0.78,5.5,3.2,1.9,1131
3,2018,071200,인피니트헬스케어,Distressed,0.090232,0.370814,0.052219,0.051247,0.254563,0.062862,...,2.183762,-1.316594,1,1.75,2.47,0.72,3.4,2.9,1.5,1100
4,2019,071200,인피니트헬스케어,Distressed,0.095699,0.389559,0.068093,0.094613,0.203532,0.086217,...,2.185795,-2.328118,1,1.25,1.62,0.37,1.4,2.2,0.4,1166
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22539,2025,364950,에이아이코리아,Normal,NaN,0.244639,-0.008910,0.385823,2.143624,-0.014508,...,1.721080,NaN,0,2.50,2.90,0.40,3.5,1.1,2.1,1450
22540,2025,455180,케이지에이,Normal,NaN,-0.029277,-0.056809,0.307518,1.658611,-0.126379,...,0.444168,NaN,0,2.50,2.90,0.40,3.5,1.1,2.1,1450
22541,2025,342870,오아,Normal,NaN,0.218109,NaN,1.260477,1.185227,NaN,...,1.326381,NaN,0,2.50,2.90,0.40,3.5,1.1,2.1,1450
22542,2025,462310,뉴키즈온,Normal,NaN,0.180441,NaN,0.205047,4.207594,NaN,...,1.665656,NaN,0,2.50,2.90,0.40,3.5,1.1,2.1,1450


In [17]:
# 1. 'Target'을 제외한 칼럼 리스트 + ['Target']으로 순서 재정의
new_order = [col for col in df_final.columns if col != 'Target'] + ['Target']
df_final = df_final[new_order]

# 2. 결과 확인 (칼럼명 출력)
print("최종 칼럼 순서:", df_final.columns.tolist())

# 3. (필요 시) 다시 저장
df_final.to_csv('Bankruptcy_Dataset_For_Modeling_With_Macro.csv', index=False, encoding='utf-8-sig')

최종 칼럼 순서: ['bsns_year', 'Company_Code', 'Company_Name', 'Status', 'F1_Equity_Growth', 'F1_Retained_Earnings_Ratio', 'F1_ROA', 'F1_Debt_Ratio', 'F1_Current_Ratio', 'F1_ROE', 'F1_Interest_Coverage', 'F2_KMV_DD', 'F3_Z_Score', 'F4_M_Score', 'M_Short_Term_Rate', 'M_Long_Term_Rate', 'M_Rate_Spread', 'M_Nominal_GDP_Growth', 'M_Real_GDP_Growth', 'M_Inflation', 'M_Exchange_Rate', 'Target']


In [18]:
df_final.head(10)

,bsns_year,Company_Code,Company_Name,Status,F1_Equity_Growth,F1_Retained_Earnings_Ratio,F1_ROA,F1_Debt_Ratio,F1_Current_Ratio,F1_ROE,...,F3_Z_Score,F4_M_Score,M_Short_Term_Rate,M_Long_Term_Rate,M_Rate_Spread,M_Nominal_GDP_Growth,M_Real_GDP_Growth,M_Inflation,M_Exchange_Rate,Target
0,2015,071200,인피니트헬스케어,Distressed,NaN,0.251549,0.068161,0.251739,2.993100,0.091093,...,3.509617,NaN,1.50,2.31,0.81,5.4,2.8,0.7,1131,1
1,2016,071200,인피니트헬스케어,Distressed,0.123921,0.308238,0.079259,0.231937,3.316511,0.103193,...,3.183807,-1.008438,1.25,1.75,0.50,4.8,2.9,1.0,1160,1
2,2017,071200,인피니트헬스케어,Distressed,0.015111,0.326141,0.016567,0.038772,0.277537,0.021194,...,2.767874,-1.150158,1.50,2.28,0.78,5.5,3.2,1.9,1131,1
3,2018,071200,인피니트헬스케어,Distressed,0.090232,0.370814,0.052219,0.051247,0.254563,0.062862,...,2.183762,-1.316594,1.75,2.47,0.72,3.4,2.9,1.5,1100,1
4,2019,071200,인피니트헬스케어,Distressed,0.095699,0.389559,0.068093,0.094613,0.203532,0.086217,...,2.185795,-2.328118,1.25,1.62,0.37,1.4,2.2,0.4,1166,1
5,2020,071200,인피니트헬스케어,Distressed,0.077331,0.403254,0.055779,0.117138,0.057872,0.073406,...,2.061923,-2.650790,0.50,1.39,0.89,0.4,-0.7,0.5,1180,1
6,2021,071200,인피니트헬스케어,Distressed,0.247353,0.492117,0.157431,0.130027,0.087768,0.199818,...,2.014890,-1.772145,1.00,1.83,0.83,6.8,4.3,2.5,1144,1
7,2022,071200,인피니트헬스케어,Distressed,0.013593,0.477740,0.001076,0.142450,0.122686,0.001401,...,1.857073,-2.490894,3.25,3.12,-0.13,4.3,2.6,5.1,1292,1
8,2023,071200,인피니트헬스케어,Distressed,0.166954,0.514284,0.001919,0.233910,4.242624,0.002505,...,2.531942,-2.182270,3.50,3.65,0.15,3.6,1.4,3.6,1305,1
9,2024,071200,인피니트헬스케어,Distressed,0.349676,0.575452,0.003090,0.221895,4.682779,0.003971,...,2.465969,0.886225,3.00,3.20,0.20,4.5,2.2,2.4,1360,1


In [19]:
import pandas as pd

# ==============================================================================
# 1. 파일 로드
# ==============================================================================
# (1) 전체 데이터셋 (Feature + 거시경제지표가 포함된 최종 파일)
#     ※ 방금 전 단계에서 저장한 파일명을 사용합니다.
file_path_dataset = 'Bankruptcy_Dataset_For_Modeling_With_Macro.csv'

# (2) 기업 리스트 파일 (업로드해주신 파일)
file_path_normal = 'master_table_cleaned(2).csv'
file_path_distressed = '관리종목_상장폐지_업종포함.csv'

try:
    df_final = pd.read_csv(file_path_dataset)
    df_normal_list = pd.read_csv(file_path_normal)
    df_distressed_list = pd.read_csv(file_path_distressed)
    print("📂 모든 파일 로드 완료!")
except FileNotFoundError as e:
    print(f"❌ 파일 로드 실패: {e}")
    # 파일이 없으면 여기서 멈춥니다.

# ==============================================================================
# 2. 필터링 대상 종목코드 추출
# ==============================================================================

# (1) 종목코드 포맷 통일 (6자리 문자열, 앞자리 0 채우기)
#     - df_final의 'Company_Code'
#     - 리스트 파일들의 '종목코드'
df_final['Company_Code'] = df_final['Company_Code'].apply(lambda x: str(x).zfill(6))
df_normal_list['종목코드'] = df_normal_list['종목코드'].apply(lambda x: str(x).zfill(6))
df_distressed_list['종목코드'] = df_distressed_list['종목코드'].apply(lambda x: str(x).zfill(6))

# (2) 정상기업 리스트에서 상위 449개 추출
target_normal_codes = df_normal_list['종목코드'].iloc[:449].tolist()
print(f"   - 정상기업 타겟: {len(target_normal_codes)}개 (상위 449개)")

# (3) 부도기업 리스트 전체 추출
target_distressed_codes = df_distressed_list['종목코드'].tolist()
print(f"   - 부도기업 타겟: {len(target_distressed_codes)}개 (전체)")

# (4) 최종 필터링용 코드 리스트 합치기
#     (중복 제거를 위해 set으로 변환했다가 다시 list로)
target_codes_all = list(set(target_normal_codes + target_distressed_codes))

# ==============================================================================
# 3. 데이터 필터링 및 저장
# ==============================================================================

# df_final에서 Company_Code가 target_codes_all에 있는 행만 남김
df_filtered = df_final[df_final['Company_Code'].isin(target_codes_all)].copy()

# 결과 확인
print("-" * 50)
print(f"📊 필터링 결과")
print(f"   - 필터링 전 데이터 수: {len(df_final)}행")
print(f"   - 필터링 후 데이터 수: {len(df_filtered)}행")
print(f"   - 남은 기업 수: {df_filtered['Company_Code'].nunique()}개")
print("-" * 50)

# 기업 상태별 분포 확인
print(df_filtered['Status'].value_counts())

# 저장
save_name = 'Bankruptcy_Dataset_Final_Filtered_실험용.csv'
df_filtered.to_csv(save_name, index=False, encoding='utf-8-sig')
print(f"\n✅ 최종 필터링된 파일 저장 완료: {save_name}")

📂 모든 파일 로드 완료!
   - 정상기업 타겟: 449개 (상위 449개)
   - 부도기업 타겟: 1290개 (전체)
--------------------------------------------------
📊 필터링 결과
   - 필터링 전 데이터 수: 22544행
   - 필터링 후 데이터 수: 10090행
   - 남은 기업 수: 1000개
--------------------------------------------------
Status
Normal        6103
Distressed    3987
Name: count, dtype: int64

✅ 최종 필터링된 파일 저장 완료: Bankruptcy_Dataset_Final_Filtered_실험용.csv
